# Stack smoke test

The three things a kernel in this container has to be able to do, and the
acceptance vehicle for F014: `pytest -m docker` executes this notebook
headlessly and fails if any cell raises.

1. Import `src/` from the read-only project mount.
2. Read an interim parquet layer through the bind mount.
3. Query the warehouse **as the read-only role**, and be refused a write.

Outputs are cleared on save, per global CLAUDE.md 3.4. Run it top to bottom
after `docker compose --profile dev up -d`; anything you build here that gets
used twice belongs in `src/` with a test.

In [ ]:
import sys

import numpy as np
import pandas as pd
import pyarrow

from src.config import INTERIM_ROOT, PROJECT_ROOT

print(f"python {sys.version.split()[0]}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | pyarrow {pyarrow.__version__}")
print(f"src imports from {PROJECT_ROOT}")

The interim layers are the point of exploring here: they carry every column
the warehouse drops. Reading a qualifying session keeps this quick — a race
is ten times the rows and a few seconds through a Windows bind mount.

In [ ]:
import time

sessions = sorted((INTERIM_ROOT / "grid").glob("2024_*_Q_projection/grid.parquet"))
if not sessions:
    raise RuntimeError("no ingested session under data/interim/grid — run the pipeline first")

started = time.perf_counter()
grid = pd.read_parquet(sessions[0], columns=["driver", "lap_number", "grid_index", "speed"])
elapsed = time.perf_counter() - started

print(f"{sessions[0].parent.name}")
print(f"{len(grid):,} grid rows in {elapsed:.2f} s, {grid['driver'].nunique()} drivers")
grid.head()

The kernel connects as the **read-only** role, the same one Grafana uses. The
credentials arrive from `servicios/.env` through the container's environment,
so no connection string is ever written into a notebook. A write must fail:
exploration reads, and the pipeline owns the warehouse.

In [ ]:
import os

import psycopg

dsn = (
    f"postgresql://{os.environ['POSTGRES_READONLY_USER']}:{os.environ['POSTGRES_READONLY_PASSWORD']}"
    f"@{os.environ.get('POSTGRES_HOST', 'postgres')}:{os.environ.get('POSTGRES_PORT', '5432')}"
    f"/{os.environ['POSTGRES_DB']}"
)

with psycopg.connect(dsn, connect_timeout=10) as conn:
    sessions_loaded = conn.execute("SELECT count(*) FROM dim_session").fetchone()[0]
    try:
        conn.execute("CREATE TABLE smoke_should_not_exist (x int)")
    except psycopg.errors.InsufficientPrivilege:
        refused = True
    else:
        refused = False

if not refused:
    raise AssertionError("the kernel can write to the warehouse — wrong role")
print(f"read-only role: {sessions_loaded} sessions loaded, write refused as it should be")

All three work. From here the useful entry points are `dim_session` and
`fact_microsector` in the warehouse, and `data/interim/` for anything the star
schema does not carry.